In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("../data/synthetic")

print("=" * 70)
print("02.32 - SIMULATION DISTRIBUTION VALIDATION")
print("=" * 70)

# Load master data
customers = pd.read_csv(BASE_DIR / "customers_master.csv")
products = pd.read_csv(BASE_DIR / "products_master.csv")
stores = pd.read_csv(BASE_DIR / "stores_master.csv")

# Load all monthly sales files
sales_files = sorted(BASE_DIR.glob("sales_*.csv"))

print("\nDATASETS LOADED")
print("-" * 70)

print(f"Customers       : {len(customers):,}")
print(f"Products        : {len(products):,}")
print(f"Stores          : {len(stores):,}")
print(f"Sales files     : {len(sales_files)}")

print("\nAPPROVED DISTRIBUTION RULES TO VALIDATE")
print("-" * 70)

print("Sales category mix:")
print("  Grocery              ~30%")
print("  Fashion              ~20%")
print("  Home & Kitchen       ~15%")
print("  Beauty & Personal    ~15%")
print("  Electronics          ~10%")
print("  Sports & Fitness     ~10%")

print("\nConcentration rules:")
print("  Top 20% stores    -> ~35%-40% of revenue")
print("  Top 20% products  -> ~60%-65% of revenue")
print("  Top 10% customers -> ~35%-40% of identified-customer revenue")

print("\n02.32 Step 1 preparation: PASSED")

02.32 - SIMULATION DISTRIBUTION VALIDATION

DATASETS LOADED
----------------------------------------------------------------------
Customers       : 10,000
Products        : 2,500
Stores          : 60
Sales files     : 32

APPROVED DISTRIBUTION RULES TO VALIDATE
----------------------------------------------------------------------
Sales category mix:
  Grocery              ~30%
  Fashion              ~20%
  Home & Kitchen       ~15%
  Beauty & Personal    ~15%
  Electronics          ~10%
  Sports & Fitness     ~10%

Concentration rules:
  Top 20% stores    -> ~35%-40% of revenue
  Top 20% products  -> ~60%-65% of revenue
  Top 10% customers -> ~35%-40% of identified-customer revenue

02.32 Step 1 preparation: PASSED


In [2]:
# ============================================================
# 02.32 - STEP 2 - SALES CATEGORY DISTRIBUTION
# FIXED USING 02.10 PRODUCT CATEGORY CONTROL PLAN
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path("..")

INPUT_SALES_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "product_category_adjusted"
    / "sales_remapped"
)

CONTROL_PLAN_FILE = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "product_category_adjusted"
    / "product_category_control_plan.csv"
)

PRODUCT_FILE = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "products_master.csv"
)

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "product_category_adjusted"
    / "sales_distribution_adjusted"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# APPROVED CATEGORY DISTRIBUTION
# ------------------------------------------------------------

CATEGORY_TARGETS = {
    "Grocery": 0.30,
    "Fashion": 0.20,
    "Home & Kitchen": 0.15,
    "Beauty & Personal Care": 0.15,
    "Electronics": 0.10,
    "Sports & Fitness": 0.10
}

# ------------------------------------------------------------
# LOAD 02.10 CONTROL PLAN
# ------------------------------------------------------------

control_plan = pd.read_csv(CONTROL_PLAN_FILE)

print("=" * 70)
print("02.32 - STEP 2 - SALES CATEGORY DISTRIBUTION")
print("=" * 70)

print("\n02.10 CONTROL PLAN LOADED")
print(f"Products in control plan : {len(control_plan):,}")

print("\nCategories available:")
print(
    control_plan["category"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# VALIDATE REQUIRED COLUMNS
# ------------------------------------------------------------

required_control_columns = [
    "product_id",
    "category"
]

missing_control_columns = [
    col
    for col in required_control_columns
    if col not in control_plan.columns
]

if missing_control_columns:
    raise ValueError(
        f"Missing columns in control plan: {missing_control_columns}"
    )

# ------------------------------------------------------------
# VALIDATE TARGET CATEGORIES
# ------------------------------------------------------------

available_categories = set(
    control_plan["category"].dropna().unique()
)

missing_categories = [
    category
    for category in CATEGORY_TARGETS
    if category not in available_categories
]

if missing_categories:
    raise ValueError(
        f"Categories missing from 02.10 control plan: {missing_categories}"
    )

# ------------------------------------------------------------
# LOAD ORIGINAL PRODUCT MASTER
# USED ONLY FOR PRICE INFORMATION
# ------------------------------------------------------------

products = pd.read_csv(PRODUCT_FILE)

price_lookup = (
    products
    .drop_duplicates("product_id")
    .set_index("product_id")["unit_price"]
    .to_dict()
)

cost_lookup = (
    products
    .drop_duplicates("product_id")
    .set_index("product_id")["cost_price"]
    .to_dict()
)

# ------------------------------------------------------------
# FIND SALES FILES
# ------------------------------------------------------------

sales_files = sorted(
    INPUT_SALES_DIR.glob("sales_*.csv")
)

print(f"\nAdjusted sales files found : {len(sales_files)}")

if len(sales_files) != 32:
    raise ValueError(
        f"Expected 32 sales files but found {len(sales_files)}"
    )

# ------------------------------------------------------------
# TOTAL SALES ROWS
# ------------------------------------------------------------

total_sales_rows = 0

for file in sales_files:
    temp = pd.read_csv(
        file,
        usecols=["sales_line_id"]
    )
    total_sales_rows += len(temp)

print(
    f"Total sales rows           : "
    f"{total_sales_rows:,}"
)

# ------------------------------------------------------------
# TARGET ROW COUNTS
# ------------------------------------------------------------

target_counts = {
    category: int(
        round(total_sales_rows * percentage)
    )
    for category, percentage
    in CATEGORY_TARGETS.items()
}

# Make sure total exactly matches sales rows

difference = (
    total_sales_rows
    - sum(target_counts.values())
)

if difference != 0:

    largest_category = max(
        CATEGORY_TARGETS,
        key=CATEGORY_TARGETS.get
    )

    target_counts[largest_category] += difference

print("\nTARGET SALES ROW COUNTS")
print("-" * 60)

for category, count in target_counts.items():

    percentage = (
        count
        / total_sales_rows
        * 100
    )

    print(
        f"{category:<25}: "
        f"{count:,} "
        f"({percentage:.2f}%)"
    )

print(
    f"\nTarget total              : "
    f"{sum(target_counts.values()):,}"
)

# ------------------------------------------------------------
# CREATE PRODUCT POOLS FROM 02.10 CONTROL PLAN
# ------------------------------------------------------------

category_product_pools = {}

for category in CATEGORY_TARGETS:

    pool = (
        control_plan[
            control_plan["category"] == category
        ]["product_id"]
        .astype(str)
        .unique()
    )

    if len(pool) == 0:
        raise ValueError(
            f"No products available for category: {category}"
        )

    category_product_pools[category] = pool

    print(
        f"{category:<25}: "
        f"{len(pool):,} products available"
    )

# ------------------------------------------------------------
# CREATE CATEGORY ASSIGNMENTS
# ------------------------------------------------------------

category_assignment = []

for category, count in target_counts.items():

    category_assignment.extend(
        [category] * count
    )

category_assignment = np.array(
    category_assignment
)

rng = np.random.default_rng(42)

rng.shuffle(category_assignment)

print(
    f"\nCategory assignments created : "
    f"{len(category_assignment):,}"
)

# ------------------------------------------------------------
# PROCESS SALES FILES
# ------------------------------------------------------------

global_position = 0

for file in sales_files:

    print(
        f"\nProcessing: {file.name}"
    )

    df = pd.read_csv(file)

    # Your source files use these names.
    # Rename only inside the dataframe.

    df = df.rename(
        columns={
            "sales_channel": "channel",
            "order_status": "status"
        }
    )

    expected_columns = [
        "sales_line_id",
        "order_id",
        "order_date",
        "order_time",
        "customer_id",
        "product_id",
        "store_id",
        "salesperson_id",
        "promotion_id",
        "quantity",
        "unit_selling_price",
        "discount_pct",
        "channel",
        "payment_method",
        "status"
    ]

    missing_columns = [
        col
        for col in expected_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns in {file.name}: "
            f"{missing_columns}"
        )

    row_count = len(df)

    # --------------------------------------------------------
    # CATEGORY ASSIGNMENTS
    # --------------------------------------------------------

    file_categories = category_assignment[
        global_position:
        global_position + row_count
    ]

    if len(file_categories) != row_count:
        raise ValueError(
            f"Category assignment mismatch in {file.name}"
        )

    global_position += row_count

    # --------------------------------------------------------
    # SELECT PRODUCTS
    # --------------------------------------------------------

    new_product_ids = np.empty(
        row_count,
        dtype=object
    )

    for category in CATEGORY_TARGETS:

        mask = (
            file_categories == category
        )

        count = int(mask.sum())

        if count == 0:
            continue

        pool = category_product_pools[
            category
        ]

        new_product_ids[mask] = rng.choice(
            pool,
            size=count,
            replace=True
        )

    # --------------------------------------------------------
    # ASSIGN PRODUCT IDs
    # --------------------------------------------------------

    df["product_id"] = new_product_ids

    # --------------------------------------------------------
    # UPDATE PRICE FROM PRODUCT MASTER
    # --------------------------------------------------------

    df["unit_selling_price"] = (
        df["product_id"]
        .map(price_lookup)
    )

    df["product_cost_price"] = (
        df["product_id"]
        .map(cost_lookup)
    )

    # --------------------------------------------------------
    # CHECK PRICE LOOKUP
    # --------------------------------------------------------

    missing_prices = (
        df["unit_selling_price"]
        .isna()
        .sum()
    )

    if missing_prices > 0:
        raise ValueError(
            f"{file.name}: "
            f"{missing_prices:,} product prices missing"
        )

    # --------------------------------------------------------
    # SAVE
    # --------------------------------------------------------

    output_file = (
        OUTPUT_DIR / file.name
    )

    df.to_csv(
        output_file,
        index=False
    )

print("\n" + "=" * 70)
print("02.32 Step 2 processing completed.")
print("=" * 70)

print(
    f"Files created : {len(sales_files)}"
)

print(
    f"Rows processed: {global_position:,}"
)

print(
    f"Output folder : {OUTPUT_DIR}"
)

02.32 - STEP 2 - SALES CATEGORY DISTRIBUTION

02.10 CONTROL PLAN LOADED
Products in control plan : 1,500

Categories available:
category
Beauty & Personal Care    200
Electronics               230
Fashion                   300
Grocery                   300
Home & Kitchen            260
Sports & Fitness          210
Name: count, dtype: int64

Adjusted sales files found : 32
Total sales rows           : 2,118,200

TARGET SALES ROW COUNTS
------------------------------------------------------------
Grocery                  : 635,460 (30.00%)
Fashion                  : 423,640 (20.00%)
Home & Kitchen           : 317,730 (15.00%)
Beauty & Personal Care   : 317,730 (15.00%)
Electronics              : 211,820 (10.00%)
Sports & Fitness         : 211,820 (10.00%)

Target total              : 2,118,200
Grocery                  : 300 products available
Fashion                  : 300 products available
Home & Kitchen           : 260 products available
Beauty & Personal Care   : 200 products availa

In [3]:
# ============================================================
# 02.32 - STEP 3 - VALIDATE SALES CATEGORY DISTRIBUTION
# ============================================================

from pathlib import Path
import pandas as pd

BASE_DIR = Path("..")

SALES_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "product_category_adjusted"
    / "sales_distribution_adjusted"
)

CONTROL_PLAN_FILE = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "product_category_adjusted"
    / "product_category_control_plan.csv"
)

CATEGORY_TARGETS = {
    "Grocery": 30.0,
    "Fashion": 20.0,
    "Home & Kitchen": 15.0,
    "Beauty & Personal Care": 15.0,
    "Electronics": 10.0,
    "Sports & Fitness": 10.0
}

# ------------------------------------------------------------
# LOAD PRODUCT CATEGORY CONTROL PLAN
# ------------------------------------------------------------

control_plan = pd.read_csv(CONTROL_PLAN_FILE)

product_category_lookup = (
    control_plan[
        ["product_id", "category"]
    ]
    .drop_duplicates("product_id")
)

product_category_lookup["product_id"] = (
    product_category_lookup["product_id"]
    .astype(str)
)

# ------------------------------------------------------------
# LOAD ALL SALES FILES
# ------------------------------------------------------------

sales_files = sorted(
    SALES_DIR.glob("sales_*.csv")
)

print("=" * 70)
print("02.32 - STEP 3 - SALES CATEGORY DISTRIBUTION VALIDATION")
print("=" * 70)

print(
    f"\nSales files found : {len(sales_files)}"
)

total_rows = 0
category_counts = {}

for file in sales_files:

    df = pd.read_csv(
        file,
        usecols=["product_id"]
    )

    df["product_id"] = (
        df["product_id"]
        .astype(str)
    )

    df = df.merge(
        product_category_lookup,
        on="product_id",
        how="left"
    )

    missing_categories = df["category"].isna().sum()

    if missing_categories > 0:
        raise ValueError(
            f"{file.name}: "
            f"{missing_categories} products have no category"
        )

    counts = df["category"].value_counts()

    for category, count in counts.items():
        category_counts[category] = (
            category_counts.get(category, 0)
            + int(count)
        )

    total_rows += len(df)

# ------------------------------------------------------------
# CREATE VALIDATION TABLE
# ------------------------------------------------------------

validation_rows = []

for category, expected_pct in CATEGORY_TARGETS.items():

    actual_rows = category_counts.get(
        category,
        0
    )

    actual_pct = (
        actual_rows / total_rows * 100
    )

    difference_pct = (
        actual_pct - expected_pct
    )

    validation_rows.append({
        "category": category,
        "actual_rows": actual_rows,
        "actual_pct": round(actual_pct, 4),
        "expected_pct": expected_pct,
        "difference_pct": round(
            difference_pct,
            4
        )
    })

validation = pd.DataFrame(
    validation_rows
)

# ------------------------------------------------------------
# DISPLAY RESULT
# ------------------------------------------------------------

print("\nFINAL SALES CATEGORY DISTRIBUTION")
print("-" * 80)

print(
    validation.to_string(
        index=False
    )
)

print(
    f"\nTotal sales rows checked : "
    f"{total_rows:,}"
)

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

# Because rows are integer counts, allow a very small
# percentage difference from the exact target.

TOLERANCE = 0.01

validation_passed = (
    validation["difference_pct"]
    .abs()
    .le(TOLERANCE)
    .all()
)

if validation_passed:

    print(
        "\n02.32 Step 3 validation: PASSED"
    )

else:

    print(
        "\n02.32 Step 3 validation: "
        "NEEDS REVIEW"
    )

02.32 - STEP 3 - SALES CATEGORY DISTRIBUTION VALIDATION

Sales files found : 32

FINAL SALES CATEGORY DISTRIBUTION
--------------------------------------------------------------------------------
              category  actual_rows  actual_pct  expected_pct  difference_pct
               Grocery       635460        30.0          30.0             0.0
               Fashion       423640        20.0          20.0             0.0
        Home & Kitchen       317730        15.0          15.0             0.0
Beauty & Personal Care       317730        15.0          15.0             0.0
           Electronics       211820        10.0          10.0             0.0
      Sports & Fitness       211820        10.0          10.0             0.0

Total sales rows checked : 2,118,200

02.32 Step 3 validation: PASSED


In [4]:
# ============================================================
# 02.32 - STEP 4 - SAVE DISTRIBUTION VALIDATION REPORT
# ============================================================

from pathlib import Path

BASE_DIR = Path("..")

REPORT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORT_FILE = (
    REPORT_DIR
    / "simulation_distribution_validation_report.csv"
)

# Save the validation result created in Step 3
validation.to_csv(
    REPORT_FILE,
    index=False
)

print("=" * 70)
print("02.32 - STEP 4 - SAVE DISTRIBUTION VALIDATION REPORT")
print("=" * 70)

print("\nValidation report:")
print(
    validation.to_string(
        index=False
    )
)

print(
    f"\nReport saved:"
)

print(
    REPORT_FILE
)

# ------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------

if (
    validation["difference_pct"]
    .abs()
    .le(0.01)
    .all()
):

    print(
        "\n02.32 Step 4 validation: PASSED"
    )

else:

    print(
        "\n02.32 Step 4 validation: NEEDS REVIEW"
    )

02.32 - STEP 4 - SAVE DISTRIBUTION VALIDATION REPORT

Validation report:
              category  actual_rows  actual_pct  expected_pct  difference_pct
               Grocery       635460        30.0          30.0             0.0
               Fashion       423640        20.0          20.0             0.0
        Home & Kitchen       317730        15.0          15.0             0.0
Beauty & Personal Care       317730        15.0          15.0             0.0
           Electronics       211820        10.0          10.0             0.0
      Sports & Fitness       211820        10.0          10.0             0.0

Report saved:
..\data\synthetic\simulation_distribution_validation_report.csv

02.32 Step 4 validation: PASSED


In [5]:
from pathlib import Path
import pandas as pd

REPORT_FILE = Path("../data/synthetic/simulation_distribution_validation_report.csv")

report = pd.read_csv(REPORT_FILE)

print("=" * 70)
print("02.32 - STEP 5 - FINAL SIMULATION DISTRIBUTION VALIDATION")
print("=" * 70)

print("\nValidation Report:")
print(report.to_string(index=False))

# Validate required columns
required_columns = [
    "category",
    "actual_rows",
    "actual_pct",
    "expected_pct",
    "difference_pct"
]

missing_columns = [
    col for col in required_columns
    if col not in report.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# Validate differences
max_difference = report["difference_pct"].abs().max()

# Validate percentages
total_actual_pct = report["actual_pct"].sum()
total_expected_pct = report["expected_pct"].sum()

print("\nTotal actual percentage   :", total_actual_pct)
print("Total expected percentage :", total_expected_pct)
print("Maximum percentage difference :", max_difference)

if (
    abs(total_actual_pct - 100) < 0.01
    and abs(total_expected_pct - 100) < 0.01
    and max_difference < 0.01
):
    print("\n02.32 Step 5 validation: PASSED")
    print("Simulation distributions are VALID.")
else:
    print("\n02.32 Step 5 validation: NEEDS REVIEW")

02.32 - STEP 5 - FINAL SIMULATION DISTRIBUTION VALIDATION

Validation Report:
              category  actual_rows  actual_pct  expected_pct  difference_pct
               Grocery       635460        30.0          30.0             0.0
               Fashion       423640        20.0          20.0             0.0
        Home & Kitchen       317730        15.0          15.0             0.0
Beauty & Personal Care       317730        15.0          15.0             0.0
           Electronics       211820        10.0          10.0             0.0
      Sports & Fitness       211820        10.0          10.0             0.0

Total actual percentage   : 100.0
Total expected percentage : 100.0
Maximum percentage difference : 0.0

02.32 Step 5 validation: PASSED
Simulation distributions are VALID.
